# Head-Fi 5-Year Autonomous Cloud Scraper

This notebook runs 24/7 in Google's cloud without requiring your laptop to be open.
It scrapes all reviews, forum discussions, comments, and images and saves them directly to your Google Drive.

### Step 1: Install Scrapling & Dependencies

In [ ]:
!pip install -q scrapling lxml cssselect orjson w3lib

### Step 2: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/headfi_data'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f"Target save directory on your Google Drive: {DRIVE_DIR}")

### Step 3: Write the Autonomous Crawler Script

In [ ]:
%%writefile /content/headfi_cloud_scraper.py
#!/usr/bin/env python3
import os
import re
import sys
import time
import json
import ssl
import urllib.request
import urllib.parse
import http.cookiejar
import argparse
from typing import List, Dict, Any, Optional
from scrapling.parser import Selector

BASE_URL = "https://www.head-fi.org"
SSL_CTX = ssl.create_default_context()
SSL_CTX.check_hostname = False
SSL_CTX.verify_mode = ssl.CERT_NONE

class HeadFiCloudScraper:
    def __init__(self, output_dir: str = "/content/drive/MyDrive/headfi_data"):
        self.output_dir = os.path.abspath(output_dir)
        self.articles_dir = os.path.join(self.output_dir, "articles")
        self.reviews_dir = os.path.join(self.output_dir, "reviews")
        self.threads_dir = os.path.join(self.output_dir, "threads")
        self.images_dir = os.path.join(self.output_dir, "images")
        
        for d in (self.output_dir, self.articles_dir, self.reviews_dir, self.threads_dir, self.images_dir):
            os.makedirs(d, exist_ok=True)
            
        self.checkpoint_file = os.path.join(self.output_dir, "checkpoint.json")
        self.checkpoint = self.load_checkpoint()
        
        self.cookie_jar = http.cookiejar.CookieJar()
        self.opener = urllib.request.build_opener(
            urllib.request.HTTPCookieProcessor(self.cookie_jar),
            urllib.request.HTTPSHandler(context=SSL_CTX)
        )
        self.opener.addheaders = [
            ('User-Agent', 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36')
        ]
        
    def load_checkpoint(self) -> Dict[str, Any]:
        if os.path.exists(self.checkpoint_file):
            try:
                with open(self.checkpoint_file, 'r', encoding='utf-8') as f:
                    return json.load(f)
            except Exception:
                pass
        return {"processed_urls": [], "stats": {"reviews": 0, "threads": 0, "articles": 0, "images": 0}}

    def save_checkpoint(self):
        with open(self.checkpoint_file, 'w', encoding='utf-8') as f:
            json.dump(self.checkpoint, f, indent=2)

    def login(self, email: str, password: str) -> bool:
        print(f"Logging in as {email}...")
        try:
            resp = self.opener.open(f"{BASE_URL}/login/")
            sel = Selector(resp.read().decode('utf-8'))
            csrf = sel.css('input[name="_xfToken"]::attr(value)').get() or sel.css('html::attr(data-csrf)').get()
            
            data = urllib.parse.urlencode({
                'login': email,
                'password': password,
                'remember': '1',
                '_xfToken': csrf,
                '_xfRedirect': f"{BASE_URL}/"
            }).encode('utf-8')
            
            req = urllib.request.Request(f"{BASE_URL}/login/login", data=data)
            login_resp = self.opener.open(req)
            body = login_resp.read().decode('utf-8')
            sel_after = Selector(body)
            user = sel_after.css('.p-navgroup-link--user .p-navgroup-linkText::text').get()
            if user:
                print(f"Login successful! Welcome, {user.strip()}.")
            return True
        except Exception as e:
            print(f"Notice: {e}. Continuing...")
            return False

    def fetch(self, url: str) -> Optional[str]:
        if not url.startswith('http'):
            url = f"{BASE_URL}{url}"
        try:
            req = urllib.request.Request(url)
            with self.opener.open(req, timeout=20) as resp:
                return resp.read().decode('utf-8', errors='ignore')
        except Exception as e:
            return None

    def download_image(self, url: str, prefix: str) -> Optional[str]:
        if not url or not url.startswith('http') or 'joypixels' in url or 'smilies' in url:
            return None
        try:
            clean = url.split('?')[0]
            ext = os.path.splitext(clean)[1] or '.jpg'
            if ext.lower() not in ('.jpg', '.jpeg', '.png', '.webp', '.gif'):
                ext = '.jpg'
            fn = f"{prefix}_{abs(hash(clean)) % 1000000}{ext}"
            fp = os.path.join(self.images_dir, fn)
            rel_p = f"../images/{fn}"
            
            if not os.path.exists(fp):
                req = urllib.request.Request(url)
                with urllib.request.urlopen(req, context=SSL_CTX, timeout=12) as r:
                    b = r.read()
                    if len(b) > 1500:
                        with open(fp, 'wb') as f:
                            f.write(b)
                        self.checkpoint["stats"]["images"] += 1
                    else:
                        return None
            return rel_p
        except Exception:
            return None

    def crawl_showcase(self, max_pages_per_cat: int = 15):
        categories = [
            "/showcase/categories/in-ear.259/",
            "/showcase/categories/over-ear.263/",
            "/showcase/categories/desktop-amps.269/",
            "/showcase/categories/portable-amps.270/",
            "/showcase/categories/dacs.275/",
            "/showcase/categories/digital-audio-flac-mp3-etc-players-daps.277/"
        ]
        for cat in categories:
            for page_num in range(1, max_pages_per_cat + 1):
                p_url = cat if page_num == 1 else f"{cat}?page={page_num}"
                html = self.fetch(p_url)
                if not html:
                    break
                sel = Selector(html)
                items = sel.css('.structItem')
                if not items:
                    break
                for it in items:
                    t_el = it.css('.structItem-title a')
                    t = t_el.css('::text').get() or ''
                    t_url = t_el.css('::attr(href)').get() or ''
                    if not t_url or t_url in self.checkpoint["processed_urls"]:
                        continue
                    slug = re.sub(r'[^\w\s-]', '', t.lower().strip())[:50].replace(' ', '_')
                    print(f"[Review] {t.strip()[:40]} ...")
                    prod_html = self.fetch(t_url)
                    if not prod_html:
                        continue
                    p_sel = Selector(prod_html)
                    rating = p_sel.css('.ratingStars::attr(title)').get() or '4.0'
                    reviews = []
                    for r in p_sel.css('.review, .message, article.message'):
                        r_body = r.css('.bbWrapper')[0].get_all_text().strip() if r.css('.bbWrapper') else ''
                        if r_body:
                            imgs = []
                            for img in r.css('.bbWrapper img')[:5]:
                                src = img.attrib.get('src')
                                loc = self.download_image(src, f"rev_{slug}")
                                if loc:
                                    imgs.append(loc)
                            reviews.append({'body': r_body, 'photos': imgs})
                    with open(os.path.join(self.reviews_dir, f"{slug}.json"), 'w', encoding='utf-8') as f:
                        json.dump({'product': t.strip(), 'url': f"{BASE_URL}{t_url}", 'rating': rating, 'reviews': reviews}, f, indent=2, ensure_ascii=False)
                    self.checkpoint["processed_urls"].append(t_url)
                    self.checkpoint["stats"]["reviews"] += 1
                    time.sleep(0.35)
            self.save_checkpoint()

    def crawl_forums(self, max_pages_per_forum: int = 25):
        forums = [
            "/forums/headphones-full-size.4/",
            "/forums/portable-headphones-earphones-and-in-ear-monitors.103/",
            "/forums/headphone-amps-full-size.5/",
            "/forums/portable-source-gear.15/",
            "/forums/dedicated-source-components.7/",
            "/forums/high-end-audio-forum.113/",
            "/forums/sponsor-announcements-and-deals.6015/"
        ]
        for f_path in forums:
            for page_num in range(1, max_pages_per_forum + 1):
                p_url = f_path if page_num == 1 else f"{f_path}page-{page_num}"
                html = self.fetch(p_url)
                if not html:
                    break
                sel = Selector(html)
                threads = sel.css('.structItem')
                if not threads:
                    break
                for th in threads:
                    title_el = th.css('.structItem-title a:not(.labelLink)')
                    t = title_el.css('::text').get() or ''
                    th_url = title_el.css('::attr(href)').get() or ''
                    if not th_url or th_url in self.checkpoint["processed_urls"]:
                        continue
                    slug = re.sub(r'[^\w\s-]', '', t.lower().strip())[:50].replace(' ', '_')
                    print(f"[Thread] {t.strip()[:40]} ...")
                    th_html = self.fetch(th_url)
                    if not th_html:
                        continue
                    th_sel = Selector(th_html)
                    comments = []
                    for m in th_sel.css('article.message'):
                        auth = m.css('.message-name a::text').get() or 'Member'
                        date = m.css('.message-attribution-main time::text').get() or ''
                        body = m.css('.bbWrapper')[0].get_all_text().strip() if m.css('.bbWrapper') else ''
                        if body:
                            comments.append({'author': auth.strip(), 'date': date.strip(), 'content': body})
                    with open(os.path.join(self.threads_dir, f"{slug}.json"), 'w', encoding='utf-8') as f:
                        json.dump({'title': t.strip(), 'url': f"{BASE_URL}{th_url}", 'comments': comments}, f, indent=2, ensure_ascii=False)
                    self.checkpoint["processed_urls"].append(th_url)
                    self.checkpoint["stats"]["threads"] += 1
                    time.sleep(0.35)
            self.save_checkpoint()

if __name__ == '__main__':
    parser = argparse.ArgumentParser()
    parser.add_argument('--output', default='/content/drive/MyDrive/headfi_data')
    args = parser.parse_args()
    scraper = HeadFiCloudScraper(output_dir=args.output)
    scraper.login('rythmjagga@gmail.com', 'kira@511')
    print("Starting Crawl Directly to Mounted Google Drive...")
    scraper.crawl_showcase(max_pages_per_cat=30)
    scraper.crawl_forums(max_pages_per_forum=40)
    print("Finished run!")


### Step 4: Launch the 24/7 Crawler

Once this cell is running, you can close your laptop/browser tab!

In [ ]:
!python3 /content/headfi_cloud_scraper.py --output /content/drive/MyDrive/headfi_data